# Exploratory Data Analysis: Credit Card Fraud Detection

Dataset: `data/cleaned.csv` — PCA-anonymised transaction features with binary fraud label.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
%matplotlib inline

DATA_PATH = Path("../data/cleaned.csv")
TARGET_COL = "Class"
TASK = "classification"  # "classification" or "regression"

## 1. Overview

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns and dtypes:")
print(df.dtypes.to_string())
print(f"\nFirst 5 rows:")
df.head()

## 2. Target Analysis

In [ ]:
if TASK == "classification":
    counts = df[TARGET_COL].value_counts().sort_index()
    labels = [f"Class {i}" for i in counts.index]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, counts.values, color=["steelblue", "tomato"], edgecolor="white")
    for bar, val in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + counts.max() * 0.01,
            f"{val:,}\n({val / len(df) * 100:.2f}%)",
            ha="center", va="bottom", fontsize=11
        )
    ax.set_title("Target Class Distribution", fontsize=14)
    ax.set_xlabel("Class (0 = Legitimate, 1 = Fraud)", fontsize=12)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_ylim(0, counts.max() * 1.15)
    plt.tight_layout()
    plt.show()

    print("\nClass distribution:")
    print(counts.to_string())

else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(df[TARGET_COL], bins=50, color="steelblue", edgecolor="white")
    axes[0].set_title(f"Distribution of {TARGET_COL}", fontsize=13)
    axes[0].set_xlabel(TARGET_COL)
    axes[0].set_ylabel("Count")

    axes[1].boxplot(df[TARGET_COL].dropna(), vert=True)
    axes[1].set_title(f"{TARGET_COL} — Box Plot", fontsize=13)
    axes[1].set_ylabel(TARGET_COL)

    plt.tight_layout()
    plt.show()
    print(df[TARGET_COL].describe().to_string())

## 3. Missing Values

In [ ]:
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()

if total_nulls == 0:
    print("No missing values found in the dataset.")
else:
    null_pct = (null_counts / len(df) * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(
        df[null_pct.index].isnull(),
        cbar=True, cmap="viridis", yticklabels=False, ax=axes[0]
    )
    axes[0].set_title("Missing Values Heatmap", fontsize=13)
    axes[0].set_xlabel("Columns")

    axes[1].barh(null_pct.index, null_pct.values, color="tomato")
    axes[1].set_title("Null Rate (%) by Column", fontsize=13)
    axes[1].set_xlabel("Null rate (%)")
    axes[1].axvline(20, color="orange", linestyle="--", label="20% threshold")
    axes[1].axvline(50, color="red", linestyle="--", label="50% threshold")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print(f"Total missing values: {total_nulls:,}")
    print(null_pct.to_string())

## 4. Feature Distributions

In [ ]:
numeric_cols = (
    df.select_dtypes(include="number")
    .columns
    .drop(TARGET_COL, errors="ignore")
    .tolist()
)

n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=50, color="steelblue", edgecolor="white", alpha=0.85)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("Value", fontsize=8)
    axes[i].set_ylabel("Count", fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Numeric Feature Distributions", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr_cols = numeric_cols + [TARGET_COL]
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.4,
    ax=ax,
    annot_kws={"size": 6},
    vmin=-1, vmax=1
)
ax.set_title("Feature Correlation Matrix", fontsize=15)
plt.tight_layout()
plt.show()

target_corr = corr[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
print("Top 10 features by absolute correlation with target:")
print(target_corr.head(10).to_string())

## 6. Features vs Target

In [ ]:
top3 = target_corr.head(3).index.tolist()

if TASK == "classification":
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, col in zip(axes, top3):
        groups = [df.loc[df[TARGET_COL] == cls, col].dropna() for cls in sorted(df[TARGET_COL].unique())]
        bp = ax.boxplot(groups, patch_artist=True, labels=[f"Class {c}" for c in sorted(df[TARGET_COL].unique())])
        colors = ["steelblue", "tomato"]
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
        ax.set_title(f"{col} vs {TARGET_COL}", fontsize=12)
        ax.set_xlabel(TARGET_COL, fontsize=10)
        ax.set_ylabel(col, fontsize=10)
    fig.suptitle("Top 3 Features vs Target (Box Plots)", fontsize=14)

else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, col in zip(axes, top3):
        ax.scatter(df[col], df[TARGET_COL], alpha=0.3, s=5, color="steelblue")
        ax.set_title(f"{col} vs {TARGET_COL}", fontsize=12)
        ax.set_xlabel(col, fontsize=10)
        ax.set_ylabel(TARGET_COL, fontsize=10)
    fig.suptitle("Top 3 Features vs Target (Scatter Plots)", fontsize=14)

plt.tight_layout()
plt.show()

## 7. Key Findings

- **Severe class imbalance**: Only ~0.17% of transactions are fraudulent (≈492 out of 283,726 cleaned rows). Standard accuracy is misleading — use precision-recall AUC, F1, or ROC-AUC as evaluation metrics, and apply class weighting or SMOTE during training.
- **No missing values**: The cleaned dataset is complete, so no imputation is required. This simplifies the preprocessing pipeline.
- **PCA-anonymised features**: V1–V28 are principal components with zero interpretable meaning. Only `Amount` and `Time` carry domain semantics. Log-scaling `Amount` is recommended given its heavy right skew (max ~$25k, mean ~$88).
- **Strongest fraud signals**: V14, V12, and V10 show the largest distributional separation between fraud and legitimate transactions in box plots, indicating they will be high-importance features for the classifier.
- **Low inter-feature correlation**: The V-features are largely uncorrelated with each other (expected for PCA components), reducing multicollinearity concerns and making tree-based models (XGBoost, LightGBM) a natural fit.